# Clasificación de Iris con XGBoost

Este notebook entrena un `XGBClassifier` sobre `Iris.csv`, evalúa su desempeño y registra el resultado en MLflow y Unity Catalog. Comparte todas las etapas de datos, evaluación y tracking con Random Forest.

## 1. Dependencias

La instalación del paquete local expone las utilidades comunes y las dependencias de entrenamiento. En Databricks, `%pip` puede reiniciar Python; ejecuta esta celda antes de importar módulos del proyecto.

In [0]:
%pip install ./tools
try:
    dbutils.library.restartPython()
except NameError:
    pass

## 2. Imports y configuración del modelo

La configuración compartida admite ejecución local y Databricks. XGBoost recibe etiquetas enteras producidas por el mismo `LabelEncoder` que utiliza el flujo de Random Forest; el mapping original se guarda en MLflow.

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier

from iris_mlflow_utils import build_config, evaluate_train_test, load_dataset, log_training_run, split_dataset

config = build_config(
    model_slug="xgboost",
    registered_model_name="workspace.default.iris_xgboost",
    run_name="xgboost-iris-challenger",
)
model_params = {
    "n_estimators": 100,
    "max_depth": 4,
    "learning_rate": 0.10,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "objective": "multi:softprob",
    "eval_metric": "mlogloss",
    "random_state": config.random_state,
    "n_jobs": 2,
}
print(f"Dataset: {config.dataset_path}")
print(f"Experimento: {config.experiment_name}")
print(f"Modelo registrado: {config.registered_model_name}")

## 3. Carga y validación de datos

La función común valida la estructura de `Iris.csv`, excluye el identificador `Id`, verifica que las features sean numéricas y codifica las especies. Esto garantiza que XGBoost y Random Forest reciban exactamente la misma entrada.

In [0]:
dataset = load_dataset(config.dataset_path)
print(f"Registros: {len(dataset.dataframe)}")
print(f"Features: {list(dataset.feature_columns)}")
print(f"Clases: {list(dataset.classes)}")
display(dataset.dataframe.head())

## 4. División train/test

El split es estratificado y reproducible. Mantener estos parámetros comunes permite comparar el resultado de XGBoost contra Random Forest sin introducir diferencias por la preparación de los datos.

In [0]:
split = split_dataset(dataset, config.test_size, config.random_state)
print(f"Train: {len(split.x_train)} filas | Test: {len(split.x_test)} filas")

## 5. Entrenamiento y evaluación

XGBoost construye árboles de forma secuencial y optimiza la función `mlogloss`. Los hiperparámetros se mantienen en un diccionario para que el mismo conjunto pueda registrarse en MLflow y compararse con otros runs.

In [0]:
model = XGBClassifier(num_class=len(dataset.classes), **model_params)
model.fit(split.x_train, split.y_train)
evaluations = evaluate_train_test(model, split, len(dataset.classes))
metrics = {
    f"{partition}_{name}": value
    for partition, result in evaluations.items()
    for name, value in result.metrics.items()
}
print(metrics)

## 6. Registro en MLflow y Unity Catalog

El helper común registra parámetros, tags, métricas train/test, reporte de clasificación, matriz de confusión, mapping de clases, signature, ejemplo de entrada y modelo XGBoost.

In [0]:
run_result = log_training_run(
    model=model,
    model_type="XGBoost",
    model_params=model_params,
    config=config,
    split=split,
    evaluations=evaluations,
    feature_columns=dataset.feature_columns,
    classes=dataset.classes,
)
print(f"Run ID: {run_result.run_id}")
print(f"Model URI: {run_result.model_uri}")
print(f"Modelo registrado: {run_result.registered_model_name}")
print(f"Versión registrada: {run_result.registered_model_version}")
print(f"Métrica principal: {run_result.metrics[config.primary_metric]:.4f}")

## 7. Verificación del modelo

Se carga el artefacto XGBoost desde MLflow y se predicen algunas filas de prueba. El resultado numérico se interpreta con el mapping de clases registrado en `class_mapping.json`.

In [0]:
loaded_model = mlflow.xgboost.load_model(run_result.model_uri)
predictions = loaded_model.predict(split.x_test.head(3))
print(f"Predicciones codificadas: {predictions.astype(int).tolist()}")
print(f"Clases: {list(dataset.classes)}")